# NISAR: зміни на Каховському водосховищі, 22.10 → 03.11.2026, і звірка з реальністю

**Що робить ноутбук**
1. Бере дві сцени NISAR GCOV з однаковою геометрією (трек 143, висхідний, кадр 025) з папки `Project/data`.
2. Перевіряє якість за метаданими: рівень шуму (NESZ) і радіоперешкоди (RFI) окремо для frequencyA (20 м) та frequencyB (80 м).
3. Показує обидві дати, різницю (Δ дБ) і кольоровий композит змін.
4. Звіряє радарні зміни з **незалежною реальністю**:
   * **OPERA DSWx-S1**: карти поверхневої води з Sentinel-1 за ті самі дні (22.10 і 03.11), 30 м;
   * **OPERA DSWx-HLS**: оптична вода з Landsat і Sentinel-2 (здебільшого під хмарами, тому лише для контролю);
   * **ESA WorldCover 2021** і **JRC Global Surface Water** (дно водосховища): стабільні класи поверхні для калібрувальної перевірки;
   * **погода ERA5** (Open-Meteo): опади й вологість ґрунту між датами, бо саме вони найчастіше змінюють L-band відбиття ріллі.
5. Видає підсумок із числами, а не лише картинки.

**Межі:** дві дати не дають тренду. Зміна відбиття ≠ зміна типу поверхні. Район — зона бойових дій: ноутбук рахує агреговану статистику довкілля, а **не** шукає військові об'єкти.

## 0. Налаштування: шлях до даних і вибір сцен

In [ ]:
from pathlib import Path
from datetime import datetime
import re, json, warnings, urllib.request, urllib.parse

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.ndimage import median_filter
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import mannwhitneyu
from pyproj import Transformer
import rasterio
from rasterio.warp import reproject, Resampling
from rasterio.transform import from_origin
from IPython.display import display

import os
os.environ.setdefault('LOKY_MAX_CPU_COUNT', str(os.cpu_count()))
warnings.filterwarnings('ignore', category=RuntimeWarning)
plt.rcParams.update({'figure.dpi': 110, 'axes.titlesize': 10})

# Папка з даними проєкту. Якщо проєкт переїде, код сам знайде папку data вище від ноутбука.
DATA_DIR = Path('..') / 'data'
if not (DATA_DIR / 'h5').exists():
    DATA_DIR = next(p / 'data' for p in [Path.cwd(), *Path.cwd().parents] if (p / 'data' / 'h5').exists())

TRACK_FRAME = '143_A_025'            # трек_напрям_кадр
DATE_OLD, DATE_NEW = '20261022', '20261103'
BBOX = (33.4, 46.9, 35.2, 47.8)      # W, S, E, N: AOI-1 проєкту (колишнє водосховище від Нової Каховки до Запоріжжя)
OUT_DIR = DATA_DIR / 'analysis_143_A_025'
OUT_DIR.mkdir(exist_ok=True)

def acq_time(p):
    return datetime.strptime(re.search(r'_(\d{8}T\d{6})_', p.name).group(1), '%Y%m%dT%H%M%S')

FILES = sorted(DATA_DIR.rglob(f'NISAR_L2_PR_GCOV_*_{TRACK_FRAME}_*.h5'))
by_date = {acq_time(p).strftime('%Y%m%d'): p for p in FILES}
print('Папка даних:', DATA_DIR)
print('Знайдено сцен', TRACK_FRAME, ':', sorted(by_date))
assert DATE_OLD in by_date and DATE_NEW in by_date, f'Немає потрібних дат у {DATA_DIR}: {sorted(by_date)}'
OLD, NEW = by_date[DATE_OLD], by_date[DATE_NEW]
T_OLD, T_NEW = acq_time(OLD), acq_time(NEW)
for p in (OLD, NEW):
    print(f'  {acq_time(p):%Y-%m-%d %H:%M} UTC  {p.stat().st_size / 2**30:.2f} ГіБ  {p.relative_to(DATA_DIR)}')

## 1. Якість за метаданими: шум і радіоперешкоди

NESZ — рівень шуму приймача в одиницях відбиття. Суходіл на L-band дає HH близько −8…−12 дБ, HV близько −15…−20 дБ. Якщо NESZ вищий за ці рівні, сигнал поверхні тоне в шумі. `rfiLikelihood` за описом у файлі має бути в межах [0, 1].

In [ ]:
GRID = '/science/LSAR/GCOV/grids'
CAL = '/science/LSAR/GCOV/metadata/calibrationInformation'
PAR = '/science/LSAR/GCOV/metadata/processingInformation/parameters'
CH = ('HHHH', 'HVHV')
db = lambda a: 10 * np.log10(a)

def utm_bounds(epsg, bbox=BBOX):
    t = Transformer.from_crs(4326, epsg, always_xy=True)
    X, Y = t.transform([bbox[0], bbox[0], bbox[2], bbox[2]], [bbox[1], bbox[3], bbox[1], bbox[3]])
    return min(X), min(Y), max(X), max(Y)

with h5py.File(OLD, 'r') as f:
    EPSG = int(f[f'{GRID}/frequencyB/projection'][()])
W_, S_, E_, N_ = utm_bounds(EPSG)

rows, flags = [], {}
for p in (OLD, NEW):
    with h5py.File(p, 'r') as f:
        for fr in ('frequencyA', 'frequencyB'):
            nes = f[f'{CAL}/{fr}/noiseEquivalentBackscatter']
            x, y = nes['xCoordinates'][:], nes['yCoordinates'][:]
            sel = np.ix_((y >= S_) & (y <= N_), (x >= W_) & (x <= E_))
            for pol in ('HH', 'HV'):
                a = nes[pol][:][sel]
                a = a[np.isfinite(a) & (a > 0)]
                rows.append({'дата': f'{acq_time(p):%d.%m}', 'смуга': fr, 'поляр.': pol,
                             'NESZ p10, дБ': db(np.percentile(a, 10)), 'NESZ медіана, дБ': db(np.median(a)),
                             'NESZ p90, дБ': db(np.percentile(a, 90)),
                             'rfiLikelihood': float(f[f'{CAL}/{fr}/{pol}/rfiLikelihood'][()])})
        flags[f'{acq_time(p):%d.%m}'] = {k: f[f'{PAR}/{k}'][()].decode() for k in
                                         ('noiseCorrectionApplied', 'rfiMitigationApplied', 'radiometricTerrainCorrectionApplied')}
qc = pd.DataFrame(rows)
display(qc.round(1).assign(rfiLikelihood=qc['rfiLikelihood'].map('{:.3g}'.format)))
display(pd.DataFrame(flags))

nesz_A = qc[(qc['смуга'] == 'frequencyA') & (qc['поляр.'] == 'HV')]['NESZ медіана, дБ'].max()
nesz_B = qc[(qc['смуга'] == 'frequencyB') & (qc['поляр.'] == 'HV')]['NESZ медіана, дБ'].max()
print(f'Медіанний шум HV: frequencyA {nesz_A:.1f} дБ, frequencyB {nesz_B:.1f} дБ.')
print('Основна смуга аналізу: frequencyB (80 м). frequencyA (20 м) показуємо для порівняння, бо в ній HV тоне в шумі перешкод.')

## 2. Спільна сітка 80 м і читання лише вікна AOI

frequencyB має крок 80 м. frequencyA (20 м) усереднюємо блоками 4×4 у лінійній потужності, тож обидві смуги лежать на **одній** сітці (центри збігаються точно). Сюди ж перепроєктуємо всі довідкові шари.

In [ ]:
with h5py.File(OLD, 'r') as f:
    xB, yB = f[f'{GRID}/frequencyB/xCoordinates'][:], f[f'{GRID}/frequencyB/yCoordinates'][:]
    xA, yA = f[f'{GRID}/frequencyA/xCoordinates'][:], f[f'{GRID}/frequencyA/yCoordinates'][:]
ci = np.where((xB >= W_) & (xB <= E_))[0]
ri = np.where((yB >= S_) & (yB <= N_))[0]
SB = (slice(ri.min(), ri.max() + 1), slice(ci.min(), ci.max() + 1))
X, Y = xB[SB[1]], yB[SB[0]]
H, W = len(Y), len(X)
RES = float(X[1] - X[0])
TRANSFORM = from_origin(X[0] - RES / 2, Y[0] + RES / 2, RES, RES)
EXTENT = [(X[0] - RES / 2) / 1e3, (X[-1] + RES / 2) / 1e3, (Y[-1] - RES / 2) / 1e3, (Y[0] + RES / 2) / 1e3]
assert np.isclose(xA[4 * SB[1].start:4 * SB[1].start + 4].mean(), X[0]) and np.isclose(yA[4 * SB[0].start:4 * SB[0].start + 4].mean(), Y[0])
print(f'EPSG:{EPSG}, сітка {H}×{W} пікселів по {RES:.0f} м ≈ {H * W * RES**2 / 1e6:,.0f} км²')
PIX_KM2 = RES**2 / 1e6

def read_freq(path, freq):
    """Повертає {канал: лінійна потужність} на сітці 80 м і маску валідних пікселів."""
    with h5py.File(path, 'r') as f:
        g = f[f'{GRID}/{freq}']
        if freq == 'frequencyB':
            out = {c: g[c][SB].astype('f4') for c in CH}
            m = g['mask'][SB]
            ok = (m > 0) & (m < 255) & (g['numberOfLooks'][SB] > 0)
        else:
            r0, c0, step = 4 * SB[0].start, 4 * SB[1].start, 128
            out = {c: np.empty((H, W), 'f4') for c in CH}
            ok = np.empty((H, W), bool)
            for i in range(0, H, step):                      # смугами, щоб не тримати все в пам'яті
                h = min(step, H - i)
                rs = slice(r0 + 4 * i, r0 + 4 * (i + h)); cs = slice(c0, c0 + 4 * W)
                for c in CH:
                    out[c][i:i + h] = np.nanmean(g[c][rs, cs].astype('f4').reshape(h, 4, W, 4), axis=(1, 3))
                m = g['mask'][rs, cs].reshape(h, 4, W, 4)
                ok[i:i + h] = ((m > 0) & (m < 255)).all(axis=(1, 3))
    for c in CH:
        ok &= np.isfinite(out[c]) & (out[c] > 0)
    return out, ok

def nesz_on_grid(path, freq, pol):
    with h5py.File(path, 'r') as f:
        nes = f[f'{CAL}/{freq}/noiseEquivalentBackscatter']
        x, y, a = nes['xCoordinates'][:], nes['yCoordinates'][:], nes[pol][:]
    a = db(np.where(a > 0, a, np.nan))
    ip = RegularGridInterpolator((y[::-1], x), a[::-1], bounds_error=False, fill_value=np.nan)
    YY, XX = np.meshgrid(Y, X, indexing='ij')
    return ip((YY, XX)).astype('f4')

D = {}   # D[freq][date] = dict(db=..., ok=..., nesz=...)
for fr in ('frequencyB', 'frequencyA'):
    D[fr] = {}
    for tag, p in (('old', OLD), ('new', NEW)):
        lin, ok = read_freq(p, fr)
        D[fr][tag] = {'db': {c: np.where(ok, db(np.maximum(lin[c], 1e-12)), np.nan).astype('f4') for c in CH}, 'ok': ok,
                      'nesz': {c: nesz_on_grid(p, fr, c[:2]) for c in CH}}
    print(fr, 'зчитано')

def change(fr, c, smooth=3):
    """Δ дБ (нова − стара) після медіанного фільтра 3×3, щоб приглушити спекл."""
    a, b = D[fr]['old']['db'][c], D[fr]['new']['db'][c]
    if smooth:
        a, b = median_filter(np.nan_to_num(a, nan=-40), smooth), median_filter(np.nan_to_num(b, nan=-40), smooth)
    return np.where(VALID[fr], b - a, np.nan)

# Піксель придатний, якщо валідний на обох датах і HH на обох датах хоча б на 3 дБ вище за шум
VALID, ABOVE_NOISE = {}, {}
for fr in D:
    o, n = D[fr]['old'], D[fr]['new']
    VALID[fr] = o['ok'] & n['ok']
    ABOVE_NOISE[fr] = {c: VALID[fr] & (o['db'][c] > o['nesz'][c] + 3) & (n['db'][c] > n['nesz'][c] + 3) for c in CH}
    print(f"{fr}: валідних {VALID[fr].mean():.1%}; вище шуму на 3 дБ: HH {ABOVE_NOISE[fr]['HHHH'].sum() / VALID[fr].sum():.1%}, "
          f"HV {ABOVE_NOISE[fr]['HVHV'].sum() / VALID[fr].sum():.1%} валідних пікселів")
DELTA = {fr: {c: change(fr, c) for c in CH} for fr in D}

## 3. Дві дати, різниця і кольоровий композит

Однакова шкала для обох дат. На панелі Δ: **синій** означає, що відбиття впало (поверхня стала гладкішою або вологою відкритою водою), **червоний** — що зросло (зволоження ґрунту, рослинність, шорсткість, перешкоди).
**Як читати HV:** велика розмита яскрава пляма в центрі дна (сильна 22.10, слабша 03.11) і відповідна синя «пляма» на Δ HV розміром у десятки км — це **зміна радіоперешкод**, а не поверхні. Вона не повторює межі полів чи русла, сильніша в HV, а у frequencyA (див. нижче й розділ 5) майже засвічує кадр. Ноутбук не робить висновків про джерело перешкод.

Композит: R = HH 22.10, G = B = HH 03.11. **Червоне** — було яскраво, стало темно (наприклад, нова вода). **Блакитне** — навпаки. Сіре — без змін.

In [ ]:
FR = 'frequencyB'
fig, axs = plt.subplots(2, 3, figsize=(16, 8.5), constrained_layout=True)
for i, c in enumerate(CH):
    a, b = D[FR]['old']['db'][c], D[FR]['new']['db'][c]
    lo, hi = np.nanpercentile(np.r_[a[VALID[FR]][::50], b[VALID[FR]][::50]], [2, 98])
    for j, (im, t) in enumerate(((a, T_OLD), (b, T_NEW))):
        q = axs[i, j].imshow(im, cmap='gray', vmin=lo, vmax=hi, extent=EXTENT)
        axs[i, j].set_title(f'{c[:2]} {FR[-1]}: {t:%d.%m.%Y}')
        fig.colorbar(q, ax=axs[i, j], shrink=.8, label='γ⁰, дБ')
    q = axs[i, 2].imshow(DELTA[FR][c], cmap='coolwarm', vmin=-6, vmax=6, extent=EXTENT)
    axs[i, 2].set_title(f'Δ {c[:2]}: 03.11 − 22.10'); fig.colorbar(q, ax=axs[i, 2], shrink=.8, label='Δ дБ')
for ax in axs.ravel():
    ax.set_xlabel('UTM 36N, км'); ax.tick_params(labelsize=7)
fig.savefig(OUT_DIR / 'dates_and_delta_freqB.png', dpi=130); plt.show()

def stretch(a, lo=-22, hi=-2):
    return np.clip((np.nan_to_num(a, nan=lo) - lo) / (hi - lo), 0, 1)
hh_o = median_filter(np.nan_to_num(D[FR]['old']['db']['HHHH'], nan=-40), 3)
hh_n = median_filter(np.nan_to_num(D[FR]['new']['db']['HHHH'], nan=-40), 3)
rgb = np.dstack([stretch(hh_o), stretch(hh_n), stretch(hh_n)])
fig, ax = plt.subplots(1, 2, figsize=(16, 6.5), constrained_layout=True)
ax[0].imshow(rgb, extent=EXTENT); ax[0].set_title('Композит змін HH (R = 22.10, G = B = 03.11), frequencyB')
q = ax[1].imshow(DELTA['frequencyA']['HHHH'], cmap='coolwarm', vmin=-6, vmax=6, extent=EXTENT)
ax[1].set_title('Для порівняння: Δ HH у frequencyA (перешкоди)'); fig.colorbar(q, ax=ax[1], shrink=.8, label='Δ дБ')
fig.savefig(OUT_DIR / 'change_composite.png', dpi=130); plt.show()

## 4. Довідкові шари на тій самій сітці: реальна вода, тип поверхні, дно водосховища

**DSWx-S1** (OPERA, з Sentinel-1): 0 = не вода, 1 = відкрита вода, 3 = затоплена рослинність; 250+ = маски або немає даних.
Для кожної дати NISAR беремо всі тайли DSWx-S1 у межах ±1 доби і заповнюємо мозаїку від найближчого за часом. Sentinel-1 на цих датах літав тими самими орбітами (12-денний повтор, як у NISAR), тому мозаїки двох дат порівнянні між собою.

In [ ]:
def to_grid(src_path, resampling=Resampling.nearest, nodata=255):
    dst = np.full((H, W), nodata, 'u1')
    with rasterio.open(src_path) as src:
        reproject(rasterio.band(src, 1), dst, dst_transform=TRANSFORM, dst_crs=f'EPSG:{EPSG}',
                  src_nodata=src.nodata if src.nodata is not None else nodata, dst_nodata=nodata, resampling=resampling)
    return dst

def dswx_mosaic(kind, t0, max_days=1.0):
    cand = []
    for p in (DATA_DIR / 'opera_dswx' / kind).glob('*_B01_WTR.tif'):
        t = datetime.strptime(re.search(r'_T\w{5}_(\d{8}T\d{6})Z_', p.name).group(1), '%Y%m%dT%H%M%S')
        dt = abs((t - t0).total_seconds()) / 86400
        if dt <= max_days:
            cand.append((dt, t, p))
    mos = np.full((H, W), 255, 'u1')
    src = np.full((H, W), -1, 'i2')          # хвилина доби знімка Sentinel-1 = ідентифікатор орбіти
    used = []
    for dt, t, p in sorted(cand):
        a = to_grid(p, Resampling.mode)
        fill = (mos >= 250) & (a < 250)
        if fill.any():
            mos[fill] = a[fill]; src[fill] = t.hour * 60 + t.minute
            used.append({'знімок': f'{t:%Y-%m-%d %H:%M}', 'тайл': re.search(r'_T(\w{5})_', p.name).group(1),
                         'Δt, год': round(dt * 24, 1), 'додано км²': round(fill.sum() * PIX_KM2)})
    return mos, src, pd.DataFrame(used)

REF = {}
for kind in ('s1', 'hls'):
    for tag, t0 in (('old', T_OLD), ('new', T_NEW)):
        mos, src, used = dswx_mosaic(kind, t0)
        REF[(kind, tag)] = (mos, src)
        valid = mos < 250
        print(f'DSWx-{kind.upper()} {t0:%d.%m}: покрито {valid.mean():.0%} AOI, вода {np.isin(mos, (1, 3)).sum() * PIX_KM2:,.0f} км²')
        if kind == 's1':
            display(used)

wc = to_grid(DATA_DIR / 'aux' / 'worldcover_2021_aoi.tif', Resampling.mode, nodata=0)
jrc = to_grid(DATA_DIR / 'aux' / 'jrc_occurrence_aoi.tif', Resampling.average, nodata=255)
RES_BED = (jrc >= 50) & (jrc <= 100)       # дно водосховища: вода ≥ 50 % часу 1984–2021
WC_NAMES = {10: 'дерева', 20: 'чагарники', 30: 'трави', 40: 'рілля', 50: 'забудова', 60: 'голий ґрунт', 80: 'вода', 90: 'водно-болотні'}
print(f'Дно колишнього водосховища (JRC ≥ 50 %): {RES_BED.sum() * PIX_KM2:,.0f} км²')

### Реальна вода за DSWx-S1 і її зміна між датами
Порівнюємо лише пікселі, де обидві дати знято **тією самою орбітою Sentinel-1**, інакше різниця кутів огляду дасть хибні зміни.

In [ ]:
s1o, src_o = REF[('s1', 'old')]
s1n, src_n = REF[('s1', 'new')]
REF_OK = (s1o < 250) & (s1n < 250) & (src_o == src_n)
w_o, w_n = np.isin(s1o, (1, 3)), np.isin(s1n, (1, 3))
REF_CHANGE = np.select([REF_OK & w_o & w_n, REF_OK & ~w_o & ~w_n, REF_OK & ~w_o & w_n, REF_OK & w_o & ~w_n], [1, 2, 3, 4], 0)
REF_LABELS = {1: 'стабільна вода', 2: 'стабільна суша', 3: 'нова вода', 4: 'вода зникла'}
ref_tab = pd.DataFrame({'клас': list(REF_LABELS.values()),
                        'км²': [(REF_CHANGE == k).sum() * PIX_KM2 for k in REF_LABELS],
                        'з них на дні водосховища, км²': [((REF_CHANGE == k) & RES_BED).sum() * PIX_KM2 for k in REF_LABELS]})
display(ref_tab.round(1))

from matplotlib.colors import ListedColormap
cm = ListedColormap(['#ffffff', '#1f5fbf', '#e8e2cf', '#18b0e8', '#d9480f'])
fig, axs = plt.subplots(1, 3, figsize=(17, 5.6), constrained_layout=True)
for ax, m, t in ((axs[0], s1o, T_OLD), (axs[1], s1n, T_NEW)):
    ax.imshow(np.select([m >= 250, np.isin(m, (1, 3))], [0, 1], 2), cmap=ListedColormap(['#ffffff', '#1f5fbf', '#e8e2cf']), vmin=0, vmax=2, extent=EXTENT)
    ax.set_title(f'DSWx-S1: вода {t:%d.%m.%Y} (біле — немає даних)')
axs[2].imshow(REF_CHANGE, cmap=cm, vmin=0, vmax=4, extent=EXTENT, interpolation='nearest')
axs[2].set_title('DSWx-S1: зміна (блакитне — нова вода, помаранчеве — зникла)')
fig.savefig(OUT_DIR / 'reference_water_dswx_s1.png', dpi=130); plt.show()

## 5. Вода як «зонд» перешкод: який реальний рівень шуму?

Справжня гладка вода на L-band дуже темна (HH нижче −20 дБ) і майже не деполяризує, тож HV у неї ще на 6–10 дБ нижче за HH. Тому виміряне над водою відбиття — це фактично **рівень шуму плюс перешкоди** у цьому місці. Якщо над водою HH ≈ HV і обидва значно вищі за NESZ із метаданих, сцена обмежена перешкодами, а не шумом приладу.

In [ ]:
from scipy.ndimage import uniform_filter
OPEN_CORE = (s1o == 1) & (s1n == 1) & (uniform_filter(((s1o == 1) & (s1n == 1)).astype('f4'), 7) > 0.99)  # відкрита вода ≥ 3 пікселі від берега
print(f'Внутрішня відкрита вода (обидві дати, далеко від берега): {OPEN_CORE.sum() * PIX_KM2:.0f} км²')
CLS = {'вода (відкрита, далеко від берега)': OPEN_CORE, 'рілля': (wc == 40) & (REF_CHANGE == 2),
       'дерева': (wc == 10) & (REF_CHANGE == 2), 'забудова': (wc == 50) & (REF_CHANGE == 2)}
rows = []
for fr in ('frequencyB', 'frequencyA'):
    for tag, t in (('old', T_OLD), ('new', T_NEW)):
        d = D[fr][tag]
        for name, m in CLS.items():
            m = m & VALID[fr]
            rows.append({'смуга': fr[-1], 'дата': f'{t:%d.%m}', 'клас': name, 'HH, дБ': np.median(d['db']['HHHH'][m]),
                         'HV, дБ': np.median(d['db']['HVHV'][m]), 'HH−HV, дБ': np.median(d['db']['HHHH'][m] - d['db']['HVHV'][m]),
                         'NESZ HH з метаданих, дБ': np.nanmedian(d['nesz']['HHHH'][m])})
probe = pd.DataFrame(rows)
display(probe.pivot_table(index=['смуга', 'клас'], columns='дата', values=['HH, дБ', 'HV, дБ', 'HH−HV, дБ', 'NESZ HH з метаданих, дБ']).round(1))
FLOOR = probe[probe['клас'].str.startswith('вода')].set_index(['смуга', 'дата'])
for fr in 'BA':
    r = FLOOR.loc[fr]
    print(f'frequency{fr}: над водою HH {r["HH, дБ"].mean():.1f} дБ, HV {r["HV, дБ"].mean():.1f} дБ; '
          f'це на {(r["HH, дБ"] - r["NESZ HH з метаданих, дБ"]).mean():.0f} дБ вище за NESZ із метаданих')

# Збільшений фрагмент: Дніпро між Нікополем і Енергодаром. Червоний контур — вода DSWx-S1 на ту саму дату
def win(lon0, lat0, lon1, lat1):
    t = Transformer.from_crs(4326, EPSG, always_xy=True)
    (x0, x1), (y0, y1) = t.transform([lon0, lon1], [lat0, lat1])
    c = np.searchsorted(X, sorted([x0, x1])); r = np.searchsorted(-Y, sorted([-y1, -y0]))
    return slice(*r), slice(*c)
ZW = win(34.15, 47.38, 34.95, 47.68)
ze = [X[ZW[1]][0] / 1e3, X[ZW[1]][-1] / 1e3, Y[ZW[0]][-1] / 1e3, Y[ZW[0]][0] / 1e3]
fig, axs = plt.subplots(2, 2, figsize=(15, 9), constrained_layout=True)
for j, (tag, wt, t) in enumerate((('old', w_o, T_OLD), ('new', w_n, T_NEW))):
    for i, fr in enumerate(('frequencyB', 'frequencyA')):
        ax = axs[i, j]
        ax.imshow(D[fr][tag]['db']['HHHH'][ZW], cmap='gray', vmin=-22, vmax=0, extent=ze)
        ax.contour(wt[ZW][::-1], [0.5], colors='r', linewidths=.6, extent=ze)
        ax.set_title(f'HH {fr}, {t:%d.%m}: червоне — вода за DSWx-S1')
fig.savefig(OUT_DIR / 'zoom_river_hh_vs_dswx.png', dpi=130); plt.show()

## 6. Чи бачить NISAR ту саму воду, що й Sentinel-1?

Порівнюємо два способи виділити воду на NISAR:
* **поріг HH** — класичний метод для чистих сцен;
* **класифікатор** (градієнтний бустинг) на ознаках HH, HV, HH−HV і локальній текстурі. Він може скористатися тим, що над водою HH ≈ HV.

Обидва навчаються **лише на 22.10** за DSWx-S1, а перевіряються на **03.11** без підлаштування. IoU = перетин / об'єднання: 1 означає ідеальний збіг, 0 — жодного.

In [ ]:
from sklearn.ensemble import HistGradientBoostingClassifier

def scores(pred, truth, m):
    p, t = pred[m], truth[m]
    tp, fp, fn = np.sum(p & t), np.sum(p & ~t), np.sum(~p & t)
    return {'IoU': tp / (tp + fp + fn), 'точність': tp / max(tp + fp, 1), 'повнота': tp / max(tp + fn, 1),
            'вода NISAR, км²': p.sum() * PIX_KM2, 'вода DSWx, км²': t.sum() * PIX_KM2}

def feats(fr, tag):
    d = D[fr][tag]['db']
    hh, hv = np.nan_to_num(d['HHHH'], nan=-30), np.nan_to_num(d['HVHV'], nan=-30)
    lin = 10 ** (hh / 10); mu = uniform_filter(lin, 5)
    cv = np.sqrt(np.maximum(uniform_filter(lin**2, 5) - mu**2, 0)) / np.maximum(mu, 1e-6)
    return np.stack([median_filter(hh, 3), median_filter(hv, 3), median_filter(hh - hv, 3), cv], -1).reshape(-1, 4)

WATER, rows = {}, []
rng = np.random.default_rng(0)
for fr in ('frequencyB', 'frequencyA'):
    Fo, Fn = feats(fr, 'old'), feats(fr, 'new')
    m_old, m_new = VALID[fr] & (s1o < 250), VALID[fr] & (s1n < 250)
    # 1) поріг HH, підібраний на 22.10
    grid = np.arange(-30, -5, 0.25)
    thr = grid[int(np.nanargmax([scores(Fo[:, 0].reshape(H, W) < t, w_o, m_old)['IoU'] for t in grid]))]
    # 2) класифікатор, навчений на 22.10
    idx = np.flatnonzero(m_old)
    wi, li = idx[w_o.ravel()[idx]], idx[~w_o.ravel()[idx]]
    sel = np.r_[rng.choice(wi, min(40000, wi.size), replace=False), rng.choice(li, min(120000, li.size), replace=False)]
    clf = HistGradientBoostingClassifier(max_iter=200, random_state=0).fit(Fo[sel], w_o.ravel()[sel])
    pred = {tag: clf.predict_proba(F)[:, 1].reshape(H, W) > 0.5 for tag, F in (('old', Fo), ('new', Fn))}
    WATER[fr] = pred
    for tag, F, wt, m, role in (('old', Fo, w_o, m_old, 'навчання'), ('new', Fn, w_n, m_new, 'перевірка')):
        rows.append({'смуга': fr, 'метод': f'поріг HH < {thr:.2f} дБ', 'дата': tag, 'роль': role, **scores(F[:, 0].reshape(H, W) < thr, wt, m)})
        rows.append({'смуга': fr, 'метод': 'класифікатор', 'дата': tag, 'роль': role, **scores(pred[tag], wt, m)})
water_skill = pd.DataFrame(rows)
display(water_skill.round(3))

fig, axs = plt.subplots(1, 2, figsize=(14, 5.5), constrained_layout=True)
for ax, tag, wt, s1, t in ((axs[0], 'old', w_o, s1o, T_OLD), (axs[1], 'new', w_n, s1n, T_NEW)):
    m = VALID['frequencyB'] & (s1 < 250); p = WATER['frequencyB'][tag]
    agree = np.select([~m, p & wt, p & ~wt, ~p & wt], [0, 1, 2, 3], 4)
    ax.imshow(agree, cmap=ListedColormap(['#ffffff', '#1f5fbf', '#d9480f', '#2fbf71', '#eeeeee']), vmin=0, vmax=4, extent=EXTENT, interpolation='nearest')
    ax.set_title(f'{t:%d.%m}, класифікатор B: синє — збіг, помаранчеве — лише NISAR, зелене — лише DSWx', fontsize=9)
fig.savefig(OUT_DIR / 'water_agreement_freqB.png', dpi=130); plt.show()

## 7. Головна перевірка: чи збігаються **зміни** NISAR зі змінами в реальності?

1. Розподіл Δ HH у класах змін за DSWx-S1. Якщо радар бачить реальні зміни, «нова вода» має мати від'ємну Δ, а «вода зникла» — додатну.
2. AUC: наскільки добре |Δ HH| відрізняє пікселі зі справжньою зміною від стабільних (0.5 = як монетка, 1.0 = ідеально).
3. Таблиця збігів змін води (класифікатор NISAR проти DSWx-S1).

In [ ]:
rng = np.random.default_rng(0)
def sample(mask, n=60000):
    idx = np.flatnonzero(mask)
    return idx if len(idx) <= n else rng.choice(idx, n, replace=False)

rows, box = [], {}
for fr in ('frequencyB', 'frequencyA'):
    dHH = DELTA[fr]['HHHH']; ok = VALID[fr] & REF_OK & np.isfinite(dHH)
    for k, name in REF_LABELS.items():
        z = dHH[ok & (REF_CHANGE == k)]
        rows.append({'смуга': fr, 'клас DSWx': name, 'км²': z.size * PIX_KM2, 'медіана Δ HH, дБ': np.median(z) if z.size else np.nan,
                     'p10': np.percentile(z, 10) if z.size else np.nan, 'p90': np.percentile(z, 90) if z.size else np.nan})
        if fr == 'frequencyB':
            box[name] = dHH.ravel()[sample(ok & (REF_CHANGE == k), 20000)]
    chg, stb = ok & np.isin(REF_CHANGE, (3, 4)), ok & np.isin(REF_CHANGE, (1, 2))
    if chg.sum() > 30:
        a, b = np.abs(dHH.ravel()[sample(chg)]), np.abs(dHH.ravel()[sample(stb)])
        auc = mannwhitneyu(a, b).statistic / (len(a) * len(b))
        rows.append({'смуга': fr, 'клас DSWx': f'AUC |Δ HH|: зміна проти стабільних = {auc:.3f}'})
change_vs_ref = pd.DataFrame(rows)
display(change_vs_ref.round(2))

fig, ax = plt.subplots(figsize=(9, 4))
ax.boxplot([v for v in box.values()], tick_labels=list(box), showfliers=False)
ax.axhline(0, c='k', lw=.8, ls='--'); ax.set_ylabel('Δ HH frequencyB, дБ'); ax.set_title('Зміна відбиття NISAR у класах змін води за DSWx-S1')
fig.savefig(OUT_DIR / 'delta_by_reference_class.png', dpi=130); plt.show()

fr = 'frequencyB'
ok = VALID[fr] & REF_OK
nis = np.select([WATER[fr]['old'] & WATER[fr]['new'], ~WATER[fr]['old'] & ~WATER[fr]['new'],
                 ~WATER[fr]['old'] & WATER[fr]['new'], WATER[fr]['old'] & ~WATER[fr]['new']], [1, 2, 3, 4], 0)
ct = pd.crosstab(pd.Series(REF_CHANGE[ok]).map(REF_LABELS).rename('DSWx-S1 (реальність)'),
                 pd.Series(nis[ok]).map(REF_LABELS).rename('NISAR frequencyB')) * PIX_KM2
print('Збіг класів змін води, км²:'); display(ct.round(1))

## 8. Калібрувальна перевірка: як змінилися стабільні типи поверхні

Забудова за 12 днів фізично майже не змінюється. Якщо її Δ далека від 0 дБ, це зсув калібрування або перешкоди, і тоді такий самий зсув сидить у всіх інших класах. Окремо дивимось на **дно колишнього водосховища**: там зараз молодий ліс, луки й русло Дніпра.

In [ ]:
rows = []
zones = {f'{v} (WorldCover)': wc == k for k, v in WC_NAMES.items()}
zones |= {'дно водосховища: дерева': RES_BED & (wc == 10), 'дно водосховища: трави/чагарники': RES_BED & np.isin(wc, (20, 30, 90)),
          'дно водосховища: все': RES_BED, 'поза дном: рілля': ~RES_BED & (wc == 40)}
for name, z in zones.items():
    for fr in ('frequencyB', 'frequencyA'):
        r = {'зона': name, 'смуга': fr[-1]}
        for c in CH:
            m = z & VALID[fr]; d = DELTA[fr][c][m]
            if d.size < 200:
                continue
            r |= {f'км²': m.sum() * PIX_KM2, f'Δ{c[:2]} медіана': np.median(d), f'|Δ{c[:2]}|>3 дБ, %': 100 * np.mean(np.abs(d) > 3),
                  f'{c[:2]} вище шуму, %': 100 * ABOVE_NOISE[fr][c][m].mean()}
        rows.append(r)
lc = pd.DataFrame(rows).dropna(axis=0, thresh=4)
display(lc.round(2))
built = lc[(lc['зона'].str.startswith('забудова')) & (lc['смуга'] == 'B')]['ΔHH медіана'].iloc[0]
built_A = lc[(lc['зона'].str.startswith('забудова')) & (lc['смуга'] == 'A')]['ΔHH медіана'].iloc[0]
print(f'Контроль калібрування (забудова): Δ HH = {built:+.2f} дБ у B, {built_A:+.2f} дБ у A')

## 9. Погода між знімками (ERA5 через Open-Meteo)

На L-band вологість верхнього шару ґрунту змінює відбиття ріллі на кілька дБ. Якщо між датами пройшли дощі, загальне «почервоніння» Δ — це волога, а не зміна землекористування. Потрібен інтернет. Без нього комірка просто пропускає цей крок.

In [ ]:
POINTS = {'Нова Каховка': (46.75, 33.37), 'Нікополь': (47.57, 34.40), 'Запоріжжя': (47.84, 35.14), 'центр дна (Велика Знам’янка)': (47.40, 34.50)}
weather = None
try:
    q = {'latitude': ','.join(str(v[0]) for v in POINTS.values()), 'longitude': ','.join(str(v[1]) for v in POINTS.values()),
         'start_date': (T_OLD.date() - pd.Timedelta(days=7)).isoformat(), 'end_date': T_NEW.date().isoformat(),
         'daily': 'precipitation_sum,temperature_2m_mean', 'hourly': 'soil_moisture_0_to_7cm', 'timezone': 'UTC'}
    with urllib.request.urlopen('https://archive-api.open-meteo.com/v1/archive?' + urllib.parse.urlencode(q), timeout=30) as r:
        res = json.load(r)
    res = res if isinstance(res, list) else [res]
    rows = []
    for name, js in zip(POINTS, res):
        d = pd.DataFrame(js['daily']); d['time'] = pd.to_datetime(d['time'])
        h = pd.DataFrame(js['hourly']); h['time'] = pd.to_datetime(h['time'])
        sm = lambda t: h.loc[(h['time'] - pd.Timestamp(t)).abs().idxmin(), 'soil_moisture_0_to_7cm']
        between = d[(d['time'] > pd.Timestamp(T_OLD.date())) & (d['time'] <= pd.Timestamp(T_NEW.date()))]
        rows.append({'точка': name, 'опади між датами, мм': between['precipitation_sum'].sum(),
                     'опади за 3 дні до 22.10, мм': d[(d['time'] >= pd.Timestamp(T_OLD.date()) - pd.Timedelta(days=2)) & (d['time'] <= pd.Timestamp(T_OLD.date()))]['precipitation_sum'].sum(),
                     'опади за 3 дні до 03.11, мм': d[(d['time'] >= pd.Timestamp(T_NEW.date()) - pd.Timedelta(days=2)) & (d['time'] <= pd.Timestamp(T_NEW.date()))]['precipitation_sum'].sum(),
                     'вологість ґрунту 22.10, м³/м³': sm(T_OLD), 'вологість ґрунту 03.11, м³/м³': sm(T_NEW),
                     'темп. 22.10, °C': d.loc[d['time'] == pd.Timestamp(T_OLD.date()), 'temperature_2m_mean'].iloc[0],
                     'темп. 03.11, °C': d.loc[d['time'] == pd.Timestamp(T_NEW.date()), 'temperature_2m_mean'].iloc[0]})
    weather = pd.DataFrame(rows); display(weather.round(3))
except Exception as e:
    print('Погоду не отримано (немає мережі або API недоступний):', e)

## 10. Підсумок: що змінилось і чи це правда

In [ ]:
ws = water_skill[water_skill['метод'] == 'класифікатор'].set_index(['смуга', 'дата'])['IoU']
thr_new = water_skill[(water_skill['метод'] != 'класифікатор') & (water_skill['дата'] == 'new')].set_index('смуга')['IoU']
floor_B = FLOOR.loc['B']
dsw_o, dsw_n = w_o[REF_OK].sum() * PIX_KM2, w_n[REF_OK].sum() * PIX_KM2
med = {r['клас DSWx']: r['медіана Δ HH, дБ'] for r in change_vs_ref[change_vs_ref['смуга'] == 'frequencyB'].to_dict('records') if 'AUC' not in r['клас DSWx']}
auc_txt = change_vs_ref[change_vs_ref['клас DSWx'].str.startswith('AUC')].set_index('смуга')['клас DSWx'].to_dict()
crop = lc[(lc['зона'] == 'рілля (WorldCover)') & (lc['смуга'] == 'B')].iloc[0]
lines = [
    f"1. Якість: шум HV у frequencyA ≈ {nesz_A:.0f} дБ (перешкоди, rfiLikelihood ~1e12), у frequencyB ≈ {nesz_B:.0f} дБ. Висновки робимо по frequencyB (80 м).",
    f"2. Реальна вода (DSWx-S1, однакова орбіта): {dsw_o:,.0f} км² → {dsw_n:,.0f} км² ({dsw_n - dsw_o:+,.0f} км²). "
    f"Нова вода {ref_tab.iloc[2]['км²']:.0f} км², зникла {ref_tab.iloc[3]['км²']:.0f} км².",
    f"3. Перешкоди видно прямо на воді: у frequencyB вода має HH {floor_B['HH, дБ'].mean():.1f} дБ і HV {floor_B['HV, дБ'].mean():.1f} дБ "
    f"(HH−HV лише {floor_B['HH−HV, дБ'].mean():.1f} дБ), тобто ефективний поріг шуму на ~{(floor_B['HH, дБ'] - floor_B['NESZ HH з метаданих, дБ']).mean():.0f} дБ вищий за NESZ.",
    f"4. Вода NISAR проти DSWx-S1, IoU на даті перевірки 03.11: класифікатор B {ws[('frequencyB', 'new')]:.2f}, A {ws[('frequencyA', 'new')]:.2f}; "
    f"простий поріг HH B {thr_new['frequencyB']:.2f}, A {thr_new['frequencyA']:.2f} (на чистих сценах зазвичай 0.7–0.9).",
    f"5. Δ HH (B) у класах DSWx: " + ', '.join(f'{k} {v:+.1f} дБ' for k, v in med.items()) + '.',
    f"6. Чи виділяє |Δ HH| справжні зміни води: " + '; '.join(f'{k[-1]}: {v.split("= ")[-1]}' for k, v in auc_txt.items()) + ' (0.5 — випадково).',
    f"7. Калібрування: забудова Δ HH = {built:+.2f} дБ (B), {built_A:+.2f} дБ (A). Рілля Δ HH = {crop['ΔHH медіана']:+.2f} дБ, |Δ|>3 дБ на {crop['|ΔHH|>3 дБ, %']:.0f} % площі.",
]
if weather is not None:
    lines.append(f"8. Погода: між датами випало {weather['опади між датами, мм'].min():.0f}–{weather['опади між датами, мм'].max():.0f} мм; "
                 f"вологість ґрунту {weather['вологість ґрунту 22.10, м³/м³'].mean():.2f} → {weather['вологість ґрунту 03.11, м³/м³'].mean():.2f} м³/м³.")
print('\n'.join(lines))
(OUT_DIR / 'summary.txt').write_text('\n'.join(lines), encoding='utf-8')
for name, df in (('qc', qc), ('water_skill', water_skill), ('change_vs_reference', change_vs_ref), ('landcover_change', lc)):
    df.to_csv(OUT_DIR / f'{name}.csv', index=False, encoding='utf-8-sig')
print('\nКартинки й таблиці збережено в', OUT_DIR)

## 11. Висновки (для цих двох сцен)

**1. Що «змінилося» на знімках NISAR.** Найпомітніша зміна — розмита пляма в HV (і в обох поляризаціях frequencyA) над центральною частиною дна. Це перешкоди: над чистою відкритою водою frequencyB показує HH ≈ −11 дБ і HV ≈ −13 дБ. Справжня вода дала б нижче −20 дБ із HV значно нижче HH. Отже, ефективний рівень шуму в районі русла на ~10 дБ вищий, ніж записано в метаданих (`noiseCorrectionApplied = False`, `rfiMitigationApplied = False`, `rfiLikelihood` ≫ 1).

**2. Що змінилося в реальності (DSWx-S1, Sentinel-1, ті самі дні й орбіти).** Площа відкритої води в AOI зросла приблизно з 393 до 437 км². Більшість нової води (≈ 70 з 99 км²) лежить на мілководдях дна колишнього водосховища. Це узгоджується з 9–18 мм опадів між датами. Прямокутну «нову воду» на південно-західному краю AOI варто вважати підозрілою (межа тайлу або артефакт DSWx), поки її не підтверджено оптикою.

**3. Чи бачить NISAR ці реальні зміни.** На рівні пікселів — ні. |Δ HH| відрізняє справжні зміни води від стабільних пікселів не краще за випадковість (AUC ≈ 0.52), а вода NISAR збігається з DSWx-S1 лише з IoU ≈ 0.1 навіть із класифікатором (на чистих сценах зазвичай 0.7–0.9). Причина — перешкоди з п. 1, а не похибка прив'язки: русло на NISAR і в DSWx лягає піксель у піксель (розділ 5).

**4. Де NISAR узгоджується з реальністю.** У середньому за великими класами поверхні frequencyB показує майже нульову зміну: забудова −0.1 дБ, рілля +0.3 дБ, ліс на дні −0.1 дБ. Це правдоподібно для 12 днів пізньої осені з майже незмінною вологістю ґрунту (ERA5: 0.38 → 0.36 м³/м³). frequencyA дає на забудові −0.9 дБ, тобто зсув через перешкоди.

**5. Що робити далі.**
* Працювати лише з frequencyB. Маскувати пікселі, де HH або HV не перевищує локального «водного» рівня шуму, а не NESZ із метаданих.
* Додати більше дат цього треку і брати медіану стека: перешкоди змінюються від дати до дати, поверхня — повільно.
* Порівнювати з DSWx-S1 і самим Sentinel-1 на кожну дату. Оптика (DSWx-HLS) у цей період майже вся під хмарами.
* Для демонстрації методу взяти чистий контрольний кадр (наприклад, озеро Чад, шум ≈ −26 дБ), а Каховку подати як історію «що радар не зміг побачити і чому».